# 05 - EPO（乾燥曲線ベース）+ PLS

## 前回EPO（04）との違い

| | 04（失敗） | 05（本ノート） |
|---|---|---|
| 干渉推定 | 樹種平均スペクトル | 含水率トレンド除去後の残差 |
| 問題 | 水分信号まで消えた | 水分を先に除いてから干渉を推定 |

## 原理
```
各樹種: X ≈ A + B × 含水率  （線形近似）
残差  = X - (A + B × 含水率) = 樹種固有の干渉成分
全樹種の残差をSVD → 上位k方向 = 干渉部分空間 D
X_epo = X - (X @ D.T) @ D
```

## 0. ライブラリのインポート

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

Path('../submissions').mkdir(parents=True, exist_ok=True)
print('ライブラリ読み込み完了')

ライブラリ読み込み完了


## 1. データの読み込み

In [2]:
train = pd.read_csv('../data/train_near.csv', encoding='cp932')
test  = pd.read_csv('../data/test_near.csv',  encoding='cp932')

META_COLS = ['sample number', 'species number', '樹種', '含水率']
SPEC_COLS = [c for c in train.columns if c not in META_COLS]

X_train = train[SPEC_COLS].values
X_test  = test[SPEC_COLS].values
y_train = train['含水率'].values
groups  = train['樹種'].values

y_min, y_max = y_train.min(), y_train.max()
print(f'Train : {X_train.shape}')
print(f'Test  : {X_test.shape}')

Train : (1322, 1555)
Test  : (550, 1555)


## 2. 前処理（SNV）

In [3]:
def snv(X):
    mean = X.mean(axis=1, keepdims=True)
    std  = X.std(axis=1, keepdims=True)
    return (X - mean) / std

X_train_snv = snv(X_train)
X_test_snv  = snv(X_test)
print('SNV完了')

SNV完了


## 3. 乾燥曲線ベース EPO

各樹種内で `X ≈ A + B×y` の線形トレンドを除去した残差から干渉方向を推定する。  
残差には「含水率で説明できない樹種固有の変動」だけが残る。

In [4]:
def fit_epo_drying(X_snv, y, groups, n_components=2):
    """
    乾燥曲線ベースEPO。trainデータのみから干渉方向Dを推定する。
    戻り値 D: (n_components, n_wavelengths) 正規直交行列
    """
    species_list = np.unique(groups)
    residuals_list = []

    for sp in species_list:
        mask = groups == sp
        X_sp = X_snv[mask]      # (n_sp, p)
        y_sp = y[mask]          # (n_sp,)

        # 樹種内で平均を引いてセンタリング
        y_c = y_sp - y_sp.mean()
        X_c = X_sp - X_sp.mean(axis=0)

        # 各波長に対する水分回帰係数: B = X_c.T @ y_c / (y_c @ y_c)
        denom = float(y_c @ y_c)
        if denom < 1e-10:
            residuals_list.append(X_c)
            continue
        B = X_c.T @ y_c / denom          # (p,) 水分感度ベクトル

        # 水分トレンドを除去 → 残差 = 樹種固有の干渉成分
        X_detrended = X_c - np.outer(y_c, B)  # (n_sp, p)
        residuals_list.append(X_detrended)

    all_residuals = np.vstack(residuals_list)  # (n_train, p)

    # SVD で上位 k 干渉方向を抽出
    _, _, Vt = np.linalg.svd(all_residuals, full_matrices=False)
    D = Vt[:n_components]  # (n_components, p)
    return D


def apply_epo(X, D):
    """X_epo = X - (X @ D.T) @ D"""
    return X - (X @ D.T) @ D


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


print('EPO関数定義完了')

EPO関数定義完了


## 4. n_components 感度確認（LOSO）

LOSO fold内でEPOをtrainだけからFitする（test樹種の情報を使わない）。  
⚠️ このスコアでn_componentsを選ばない → LOSOが改善していればLBで1回だけ確認

In [5]:
print('n_epo  LOSO_mean  LOSO_median')
print('-----  ---------  -----------')

for n in [0, 1, 2, 3, 5, 8]:
    species_all = np.unique(groups)
    scores = {}
    for sp in species_all:
        val_mask = groups == sp
        tr_mask  = ~val_mask
        X_tr  = X_train_snv[tr_mask]
        X_val = X_train_snv[val_mask]
        y_tr  = y_train[tr_mask]
        if n > 0:
            D = fit_epo_drying(X_tr, y_tr, groups[tr_mask], n_components=n)
            X_tr  = apply_epo(X_tr,  D)
            X_val = apply_epo(X_val, D)
        pls = PLSRegression(n_components=3)
        pls.fit(X_tr, y_tr)
        scores[sp] = rmse(y_train[val_mask], pls.predict(X_val).ravel())
    s = pd.Series(scores)
    print(f'  {n:2d}     {s.mean():6.2f}       {s.median():6.2f}')

n_epo  LOSO_mean  LOSO_median
-----  ---------  -----------
   0      21.35        15.07
   1      21.98        16.51
   2      29.65        15.45
   3      32.14        16.06
   5      33.17        16.10
   8      77.68        34.22


## 4b. LOSO 樹種別スコア（選択したnで確認）

In [6]:
N_EPO = 2  # ← 感度確認を見て設定する

species_all = np.unique(groups)
scores_base = {}
scores_epo  = {}

for sp in species_all:
    val_mask = groups == sp
    tr_mask  = ~val_mask
    X_tr, X_val = X_train_snv[tr_mask], X_train_snv[val_mask]
    y_tr = y_train[tr_mask]

    # ベースライン
    pls = PLSRegression(n_components=3)
    pls.fit(X_tr, y_tr)
    scores_base[sp] = rmse(y_train[val_mask], pls.predict(X_val).ravel())

    # EPO
    D = fit_epo_drying(X_tr, y_tr, groups[tr_mask], n_components=N_EPO)
    X_tr_e  = apply_epo(X_tr,  D)
    X_val_e = apply_epo(X_val, D)
    pls2 = PLSRegression(n_components=3)
    pls2.fit(X_tr_e, y_tr)
    scores_epo[sp] = rmse(y_train[val_mask], pls2.predict(X_val_e).ravel())

df_compare = pd.DataFrame({'baseline': scores_base, f'EPO(n={N_EPO})': scores_epo})
df_compare['diff'] = df_compare[f'EPO(n={N_EPO})'] - df_compare['baseline']
df_compare = df_compare.sort_values('baseline')
print(df_compare.round(2).to_string())
print(f'\nbaseline  mean: {df_compare["baseline"].mean():.2f}  median: {df_compare["baseline"].median():.2f}')
print(f'EPO(n={N_EPO})  mean: {df_compare[f"EPO(n={N_EPO})"].mean():.2f}  median: {df_compare[f"EPO(n={N_EPO})"].median():.2f}')

         baseline  EPO(n=2)    diff
スプルース        5.05      7.70    2.64
ベイマツ         6.51      7.51    1.00
クリ          10.83     10.88    0.05
トチ          12.70     11.19   -1.51
ウォールナット     14.19      5.64   -8.55
米ヒバ         14.41     14.54    0.13
ヒノキ         15.07     15.45    0.38
ホワイトオーク     15.55     16.32    0.77
イチョウ        18.20     16.95   -1.26
ナラ          23.21     22.64   -0.57
ウエンジ        34.62     38.55    3.93
チェリー        35.90     35.81   -0.09
ベイスギ        71.33    182.22  110.88

baseline  mean: 21.35  median: 15.07
EPO(n=2)  mean: 29.65  median: 15.45


## 5. 最終モデル・提出CSV生成

In [7]:
N_EPO = 2  # ← 上のセルと合わせる

D_final = fit_epo_drying(X_train_snv, y_train, groups, n_components=N_EPO)
X_train_epo = apply_epo(X_train_snv, D_final)
X_test_epo  = apply_epo(X_test_snv,  D_final)

pls_final = PLSRegression(n_components=3)
pls_final.fit(X_train_epo, y_train)

y_pred         = pls_final.predict(X_test_epo).ravel()
y_pred_clipped = np.clip(y_pred, y_min, y_max)

print(f'予測値（clip前）: min={y_pred.min():.2f}, max={y_pred.max():.2f}')
print(f'予測値（clip後）: min={y_pred_clipped.min():.2f}, max={y_pred_clipped.max():.2f}')

fname = f'../submissions/submission_epo_dry{N_EPO}_pls3.csv'
pd.DataFrame({
    'sample number': test['sample number'],
    '含水率': y_pred_clipped
}).to_csv(fname, index=False, header=False)
print(f'\n保存完了: {fname}')

予測値（clip前）: min=-8.45, max=181.49
予測値（clip後）: min=0.84, max=181.49

保存完了: ../submissions/submission_epo_dry2_pls3.csv
